In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [16]:
a = torch.tensor([[0,1,2,3,4]])
x = torch.tensor([[3,1,4,2,5]])
n_embd = 8
head_size = 2
block_size = 5

In [17]:
tet = nn.Embedding(6, n_embd)
pet = nn.Embedding(block_size, n_embd)
key = nn.Linear(n_embd, head_size, bias=False)
query = nn.Linear(n_embd, head_size, bias=False)

In [19]:
concat = torch.cat([tet(x), pet(a)])
add = tet(x) + pet(a)

In [21]:
kx = key(add)

In [22]:
qx = query(add)

In [23]:
tril = torch.tril(torch.ones(block_size, block_size))

In [24]:
wx = (qx @ kx.transpose(-2,-1))
wx = wx.masked_fill(tril[:block_size, :block_size] == 0, float("-inf"))
wx = F.softmax(wx, dim=-1)

In [25]:
value = nn.Linear(n_embd, head_size, bias=False)

In [26]:
vx = value(add)

In [27]:
wx @ vx

tensor([[[-1.1734, -0.3377],
         [-0.4464, -0.7260],
         [ 0.4619, -0.3723],
         [-0.5499,  0.0813],
         [-0.3892,  0.0679]]], grad_fn=<UnsafeViewBackward0>)

In [38]:
(wx @ vx).shape

torch.Size([1, 5, 2])

In [39]:
x.shape

torch.Size([1, 5])

In [33]:
def forward(inp):
    kx = key(inp)
    qx = query(inp)
    wx = (qx @ kx.transpose(-2,-1))
    wx = wx.masked_fill(tril[:block_size, :block_size] == 0, float("-inf"))
    wx = F.softmax(wx, dim=-1)

    vx = value(add)

    return wx, vx, kx

In [34]:
wt, vt, kt = forward(tet(x))

In [35]:
wp, vp, kp = forward(pet(a))

In [36]:
wp, wt, wx

(tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
          [0.2299, 0.7701, 0.0000, 0.0000, 0.0000],
          [0.4879, 0.2928, 0.2193, 0.0000, 0.0000],
          [0.2406, 0.2512, 0.2600, 0.2483, 0.0000],
          [0.2888, 0.1722, 0.1240, 0.2494, 0.1656]]],
        grad_fn=<SoftmaxBackward0>),
 tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
          [0.5126, 0.4874, 0.0000, 0.0000, 0.0000],
          [0.3095, 0.3009, 0.3896, 0.0000, 0.0000],
          [0.2186, 0.2325, 0.1417, 0.4071, 0.0000],
          [0.2012, 0.2086, 0.1975, 0.1935, 0.1993]]],
        grad_fn=<SoftmaxBackward0>),
 tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
          [0.1970, 0.8030, 0.0000, 0.0000, 0.0000],
          [0.2897, 0.2395, 0.4707, 0.0000, 0.0000],
          [0.3715, 0.1501, 0.0495, 0.4289, 0.0000],
          [0.2684, 0.1687, 0.1260, 0.2912, 0.1457]]],
        grad_fn=<SoftmaxBackward0>))

In [37]:
kp, kt, kx

(tensor([[[-0.9183,  0.1858],
          [ 0.1582, -0.2945],
          [ 0.8495, -0.5116],
          [-0.5937,  0.2950],
          [ 0.2844,  0.2550]]], grad_fn=<UnsafeViewBackward0>),
 tensor([[[-0.1920, -0.0033],
          [-0.0372, -0.3693],
          [-0.0505,  0.3718],
          [-0.7351,  0.0656],
          [ 0.0841,  0.3630]]], grad_fn=<UnsafeViewBackward0>),
 tensor([[[-1.1104,  0.1826],
          [ 0.1210, -0.6637],
          [ 0.7990, -0.1399],
          [-1.3289,  0.3607],
          [ 0.3684,  0.6180]]], grad_fn=<UnsafeViewBackward0>))